# 02 · Feature Engineering
The single **`FeaturePipeline`** turns raw listings into a model-ready matrix and is reused verbatim at inference time — eliminating train/serve skew. Here we inspect what it produces.

In [1]:
import os, sys
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 120)
%matplotlib inline

In [2]:
from src.data import load_parquet
from src.features.parser import parse_listing
from src.features.pipeline import FeaturePipeline
df = load_parquet('data/processed/training.parquet')
print('training rows:', len(df))

training rows: 10166


### The parser on real titles

In [3]:
for t in ['iPhone 13 Pro 256GB Sierra Blue batería 92%',
          'iPhone 11 64gb como nuevo con caja',
          'iPhone 15 Pro Max 1TB titanio natural']:
    print(t)
    print('  ->', {k:v for k,v in parse_listing(t,'').items() if v not in (None,False)})

iPhone 13 Pro 256GB Sierra Blue batería 92%
  -> {'model_family': 'iPhone 13 Pro', 'model_year': 2021, 'storage_gb': 256, 'color': 'sierra_blue', 'battery_pct': 92, 'is_iphone': True}
iPhone 11 64gb como nuevo con caja
  -> {'model_family': 'iPhone 11', 'model_year': 2019, 'storage_gb': 64, 'has_box': True, 'condition_label': 3, 'is_iphone': True}
iPhone 15 Pro Max 1TB titanio natural
  -> {'model_family': 'iPhone 15 Pro Max', 'model_year': 2023, 'storage_gb': 1024, 'color': 'titanium', 'is_iphone': True}


### Fit the pipeline and inspect the feature matrix

In [4]:
pipe = FeaturePipeline(tfidf_max_features=50)
X = pipe.fit_transform(df)
print('feature matrix:', X.shape)
print('NaNs after imputation:', int(X.isna().sum().sum()))
X.iloc[:3, :12]

feature matrix: (10166, 105)
NaNs after imputation: 0


,model_year,storage_gb,battery_pct,condition_label,n_photos,description_length,days_since_posted,city_tier,seller_n_listings,has_box,has_warranty,has_accessories
0,2019.0,128.0,75.0,4.0,5.0,352.0,3.523845,1.0,1.0,1.0,0.0,1.0
1,2019.0,128.0,75.5,2.0,1.0,112.0,0.265201,1.0,1.0,0.0,0.0,0.0
2,2019.0,64.0,75.5,2.0,4.0,399.0,0.577625,1.0,1.0,0.0,0.0,0.0


In [5]:
# feature groups
groups = {'one-hot model': sum(c.startswith('fam_') for c in X.columns),
          'one-hot colour': sum(c.startswith('color_') for c in X.columns),
          'tfidf': sum(c.startswith('tfidf_') for c in X.columns),
          'numeric/binary': sum(not c.startswith(('fam_','color_','tfidf_')) for c in X.columns)}
pd.Series(groups, name='n_features').to_frame()

,n_features
one-hot model,24
one-hot colour,15
tfidf,50
numeric/binary,16


### Train/inference parity
An unseen model and colour still produce the exact same columns — the guarantee that prevents train/serve skew.

In [6]:
infer = df.head(3).copy()
infer.loc[infer.index[0],'title'] = 'iPhone 99 Ultra 256GB plata'
Xi = pipe.transform(infer)
print('same columns:', list(Xi.columns)==pipe.feature_names_, '| shape', Xi.shape)

same columns: True | shape (3, 105)
